# GW150914 GWFish Fisher matrices with AMIGO networks

This notebook computes the requested AMIGO, AMIGO-5, LVK, CE, LISA, Taiji, and TianQin network Fisher matrices. AMIGO and AMIGO-5 reuse GWFish's LISA solar-orbit response while replacing the component PSD with `amigo_psd` from `test_notebook_functions.py` on a $0.01$--$10$ Hz frequency grid. The two arm-length choices are $L_{\rm AMIGO}=10^7$ m and $5\times10^7$ m.

Each unique detector Fisher matrix is evaluated once and cached; network matrices are sums of those independent detector contributions. Taiji and TianQin likewise reuse the GWFish LISA response with their repository PSDs.

In [1]:
import os
import re
import sys
import types
import importlib.machinery
import importlib.util
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'code' else cwd
DATADIR = PROJECT_ROOT / 'data' / 'gw150914_gwfish_fisher_amigo'
DATADIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib-cache'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

try:
    import lzma  # noqa: F401
except ModuleNotFoundError:
    lzma_stub = types.ModuleType('lzma')
    lzma_stub.__spec__ = importlib.machinery.ModuleSpec('lzma', loader=None)
    class LZMAFile:
        pass
    lzma_stub.LZMAFile = LZMAFile
    lzma_stub.open = lambda *args, **kwargs: None
    sys.modules['lzma'] = lzma_stub

import numpy as np
from scipy.interpolate import interp1d

if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid

import GWFish
from GWFish.modules import detection, fishermatrix, waveforms

tnf_path = PROJECT_ROOT / 'code' / 'test_notebook_functions.py'
tnf_spec = importlib.util.spec_from_file_location('test_notebook_functions', tnf_path)
tnf = importlib.util.module_from_spec(tnf_spec)
tnf_spec.loader.exec_module(tnf)

print(f'Output directory: {DATADIR}')

/home/ansonchen/multiband_cosmo/.venv/lib/python3.12/site-packages/lalsimulation/_lalsimulation_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


Output directory: /home/ansonchen/multiband_cosmo/data/gw150914_gwfish_fisher_amigo


## Source parameters

The source and four-year time to merger match the companion LISA/LGWA/LVK/CE notebook.

In [2]:
YEAR = 365.25 * 24 * 3600
TIME_TO_MERGER_YR = 4.0
OBSERVATION_START_GPS = 1735257618.0
COALESCENCE_GPS = OBSERVATION_START_GPS + TIME_TO_MERGER_YR * YEAR

GW150914 = {
    'chirp_mass': 28.1,
    'mass_ratio': 29.0 / 36.0,
    'luminosity_distance': 200.0,
    'redshift': 0.04,
    'theta_jn': 0.8,
    'ra': 3.0,
    'dec': 0.0,
    'psi': 0.4,
    'geocent_time': COALESCENCE_GPS,
    'phase': 1.5,
    'a_1': 0.10,
    'a_2': 0.05,
}

FISHER_PARAMETERS = [
    'chirp_mass', 'mass_ratio', 'luminosity_distance', 'theta_jn',
    'ra', 'dec', 'psi', 'geocent_time', 'phase', 'a_1', 'a_2',
]

print(f'Time to merger: {TIME_TO_MERGER_YR:.1f} yr')
print(f'Coalescence GPS: {COALESCENCE_GPS:.1f}')

Time to merger: 4.0 yr
Coalescence GPS: 1861488018.0


## Detector construction

The AMIGO PSD is already an equivalent strain-noise PSD. Following the requested PSD-replacement approach, it is assigned directly to every GWFish LISA-response component. The LISA orbital geometry and response implementation are retained; only the PSD and frequency grid are replaced.

In [3]:
GWFISH_ROOT = Path(GWFish.__file__).resolve().parent
PSD_DIR = GWFISH_ROOT / 'detector_psd'
TWO_CE_CONFIG = DATADIR / 'two_ce2_sites.yaml'

TWO_CE_CONFIG.write_text(f'''CE_LHO:
  lat: 46.46 * np.pi / 180.
  lon: -119.4 * np.pi / 180.
  opening_angle: np.pi / 2.
  azimuth: 126. * np.pi / 180.
  psd_path: "Path('{PSD_DIR}')"
  psd_data: CE2_psd.txt
  duty_factor: 1.0
  detector_class: earthL
  plotrange: 5, 2048, 1e-26, 1e-20
  fmin: 5
  fmax: 1024
  spacing: geometric
  npoints: 1800
  arm_length: 40000
CE_LLO:
  lat: 30.56 * np.pi / 180.
  lon: -90.77 * np.pi / 180.
  opening_angle: np.pi / 2.
  azimuth: 197.7 * np.pi / 180.
  psd_path: "Path('{PSD_DIR}')"
  psd_data: CE2_psd.txt
  duty_factor: 1.0
  detector_class: earthL
  plotrange: 5, 2048, 1e-26, 1e-20
  fmin: 5
  fmax: 1024
  spacing: geometric
  npoints: 1800
  arm_length: 40000
''')


def lisa_response_with_replaced_psd(name, psd_function, frequency=None):
    detector = detection.Detector('LISA')
    if frequency is None:
        frequency = np.asarray(detector.frequencyvector, dtype=float).squeeze()
    else:
        frequency = np.asarray(frequency, dtype=float)
        detector.frequencyvector = frequency[:, None]
    psd = np.asarray(psd_function(frequency), dtype=float)
    valid = np.isfinite(psd) & (psd > 0.0)
    if not np.all(valid):
        raise ValueError(f'Invalid PSD values for {name}')
    for component in detector.components:
        component.psd_data = np.column_stack([frequency, psd])
        component.Sn = interp1d(
            frequency, psd, bounds_error=False, fill_value=np.inf,
            assume_sorted=True,
        )
    detector.name = name
    return detector


AMIGO_FREQUENCY = np.geomspace(1.0e-2, 10.0, 2400)

DETECTOR_FACTORIES = {
    'LISA': lambda: detection.Detector('LISA'),
    'Taiji': lambda: lisa_response_with_replaced_psd(
        'Taiji',
        lambda frequency: tnf.taiji_psd(
            frequency, include_confusion=True, observation_years=TIME_TO_MERGER_YR
        ),
    ),
    'TianQin': lambda: lisa_response_with_replaced_psd(
        'TianQin',
        lambda frequency: tnf.tianqin_psd(
            frequency, observation_years=TIME_TO_MERGER_YR
        ),
    ),
    'AMIGO': lambda: lisa_response_with_replaced_psd(
        'AMIGO', lambda frequency: tnf.amigo_psd(frequency, LAMIGO=1.0e7),
        AMIGO_FREQUENCY,
    ),
    'AMIGO-5': lambda: lisa_response_with_replaced_psd(
        'AMIGO-5', lambda frequency: tnf.amigo_psd(frequency, LAMIGO=5.0e7),
        AMIGO_FREQUENCY,
    ),
    'LHO': lambda: detection.Detector('LHO'),
    'LLO': lambda: detection.Detector('LLO'),
    'VIR': lambda: detection.Detector('VIR'),
    'KAG': lambda: detection.Detector('KAG'),
    'CE-LHO': lambda: detection.Detector('CE_LHO', config=TWO_CE_CONFIG),
    'CE-LLO': lambda: detection.Detector('CE_LLO', config=TWO_CE_CONFIG),
}

DETECTOR_DISPLAY_NAMES = {
    'LISA': 'LISA', 'Taiji': 'Taiji (LISA response)',
    'TianQin': 'TianQin (LISA response)',
    'AMIGO': r'AMIGO ($L=10^7$ m; LISA response)',
    'AMIGO-5': r'AMIGO-5 ($L=5\times10^7$ m; LISA response)',
    'LHO': 'LIGO Hanford O5', 'LLO': 'LIGO Livingston O5',
    'VIR': 'Virgo O5', 'KAG': 'KAGRA O5',
    'CE-LHO': 'CE-LHO', 'CE-LLO': 'CE-LLO',
}

LVK = ('LHO', 'LLO', 'VIR', 'KAG')
TWO_CE = ('CE-LHO', 'CE-LLO')
LTT = ('LISA', 'Taiji', 'TianQin')

## Requested networks

In [4]:
NETWORKS = {
    'AMIGO': ('AMIGO',),
    'AMIGO-5': ('AMIGO-5',),
    'AMIGO + LVK': ('AMIGO',) + LVK,
    'AMIGO-5 + LVK': ('AMIGO-5',) + LVK,
    'AMIGO + 2CE': ('AMIGO',) + TWO_CE,
    'AMIGO-5 + 2CE': ('AMIGO-5',) + TWO_CE,
    'LISA + AMIGO + LVK': ('LISA', 'AMIGO') + LVK,
    'LISA + AMIGO-5 + LVK': ('LISA', 'AMIGO-5') + LVK,
    'LISA-Taiji-TianQin + AMIGO + LVK': LTT + ('AMIGO',) + LVK,
    'LISA-Taiji-TianQin + AMIGO-5 + LVK': LTT + ('AMIGO-5',) + LVK,
    'LISA + AMIGO + 2CE': ('LISA', 'AMIGO') + TWO_CE,
    'LISA + AMIGO-5 + 2CE': ('LISA', 'AMIGO-5') + TWO_CE,
    'LISA-Taiji-TianQin + AMIGO + 2CE': LTT + ('AMIGO',) + TWO_CE,
    'LISA-Taiji-TianQin + AMIGO-5 + 2CE': LTT + ('AMIGO-5',) + TWO_CE,
}

for network_name, detector_keys in NETWORKS.items():
    print(f'{network_name}: {detector_keys}')

AMIGO: ('AMIGO',)
AMIGO-5: ('AMIGO-5',)
AMIGO + LVK: ('AMIGO', 'LHO', 'LLO', 'VIR', 'KAG')
AMIGO-5 + LVK: ('AMIGO-5', 'LHO', 'LLO', 'VIR', 'KAG')
AMIGO + 2CE: ('AMIGO', 'CE-LHO', 'CE-LLO')
AMIGO-5 + 2CE: ('AMIGO-5', 'CE-LHO', 'CE-LLO')
LISA + AMIGO + LVK: ('LISA', 'AMIGO', 'LHO', 'LLO', 'VIR', 'KAG')
LISA + AMIGO-5 + LVK: ('LISA', 'AMIGO-5', 'LHO', 'LLO', 'VIR', 'KAG')
LISA-Taiji-TianQin + AMIGO + LVK: ('LISA', 'Taiji', 'TianQin', 'AMIGO', 'LHO', 'LLO', 'VIR', 'KAG')
LISA-Taiji-TianQin + AMIGO-5 + LVK: ('LISA', 'Taiji', 'TianQin', 'AMIGO-5', 'LHO', 'LLO', 'VIR', 'KAG')
LISA + AMIGO + 2CE: ('LISA', 'AMIGO', 'CE-LHO', 'CE-LLO')
LISA + AMIGO-5 + 2CE: ('LISA', 'AMIGO-5', 'CE-LHO', 'CE-LLO')
LISA-Taiji-TianQin + AMIGO + 2CE: ('LISA', 'Taiji', 'TianQin', 'AMIGO', 'CE-LHO', 'CE-LLO')
LISA-Taiji-TianQin + AMIGO-5 + 2CE: ('LISA', 'Taiji', 'TianQin', 'AMIGO-5', 'CE-LHO', 'CE-LLO')


## Fisher calculation and outputs

The reported sky area is the covariance-aware 90% two-dimensional Gaussian credible region.

In [5]:
def sky_localization_area(covariance, declination, confidence=0.90):
    ra_index = FISHER_PARAMETERS.index('ra')
    dec_index = FISHER_PARAMETERS.index('dec')
    angular_covariance = covariance[np.ix_([ra_index, dec_index], [ra_index, dec_index])]
    determinant = max(float(np.linalg.det(angular_covariance)), 0.0)
    chi_squared_contour = -2.0 * np.log1p(-confidence)
    area_sr = np.pi * chi_squared_contour * abs(np.cos(declination)) * np.sqrt(determinant)
    return area_sr, area_sr * (180.0 / np.pi) ** 2


def compute_detector_contribution(detector_key):
    detector = DETECTOR_FACTORIES[detector_key]()
    fisher, snr_squared = fishermatrix.compute_detector_fisher(
        detector, GW150914, fisher_parameters=FISHER_PARAMETERS,
        waveform_model='IMRPhenomXPHM', waveform_class=waveforms.LALFD_Waveform,
        use_duty_cycle=False,
    )
    result = {'fisher': fisher, 'snr': float(np.sqrt(snr_squared))}
    print(f"{DETECTOR_DISPLAY_NAMES[detector_key]}: SNR={result['snr']:.3f}")
    return result


required_detector_keys = list(dict.fromkeys(
    detector_key for detector_keys in NETWORKS.values() for detector_key in detector_keys
))
detector_results = {
    detector_key: compute_detector_contribution(detector_key)
    for detector_key in required_detector_keys
}


def combine_network(detector_keys):
    network_fisher = sum(
        (detector_results[key]['fisher'] for key in detector_keys),
        np.zeros((len(FISHER_PARAMETERS), len(FISHER_PARAMETERS))),
    )
    covariance, singular_values = fishermatrix.invertSVD(network_fisher)
    if covariance is None:
        raise RuntimeError(f'GWFish SVD inversion failed for {detector_keys}')
    covariance = 0.5 * (covariance + covariance.T)
    uncertainties = np.sqrt(np.clip(np.diag(covariance), 0.0, np.inf))
    sky_area_90_sr, sky_area_90_deg2 = sky_localization_area(covariance, GW150914['dec'])
    return {
        'fisher': network_fisher, 'covariance': covariance,
        'uncertainties': uncertainties, 'singular_values': np.asarray(singular_values),
        'rank': int(np.sum(np.asarray(singular_values) > 1.0e-10)),
        'network_snr': float(np.sqrt(sum(detector_results[key]['snr'] ** 2 for key in detector_keys))),
        'detector_keys': detector_keys,
        'sky_localization_90_sr': sky_area_90_sr,
        'sky_localization_90_deg2': sky_area_90_deg2,
    }


network_results = {name: combine_network(keys) for name, keys in NETWORKS.items()}

master_rows = []
for network_name, result in network_results.items():
    output_name = re.sub(r'[^a-z0-9]+', '_', network_name.lower()).strip('_')
    np.savetxt(DATADIR / f'fisher_{output_name}.txt', result['fisher'])
    np.savetxt(DATADIR / f'covariance_{output_name}.txt', result['covariance'])
    summary_path = DATADIR / f'{output_name}_summary.csv'
    with summary_path.open('w') as handle:
        handle.write('parameter,fiducial,sigma\n')
        for parameter, sigma in zip(FISHER_PARAMETERS, result['uncertainties']):
            handle.write(f'{parameter},{GW150914[parameter]},{sigma}\n')
        handle.write(f"network_snr,,{result['network_snr']}\n")
        handle.write(f"fisher_rank,,{result['rank']}\n")
        handle.write(f"sky_localization_90_sr,,{result['sky_localization_90_sr']}\n")
        handle.write(f"sky_localization_90_deg2,,{result['sky_localization_90_deg2']}\n")

    print(f"\n{network_name}: SNR={result['network_snr']:.3f}, rank={result['rank']}/{len(FISHER_PARAMETERS)}")
    for detector_key in result['detector_keys']:
        print(f"  {DETECTOR_DISPLAY_NAMES[detector_key]}: SNR={detector_results[detector_key]['snr']:.3f}")
    for parameter, sigma in zip(FISHER_PARAMETERS, result['uncertainties']):
        print(f'  sigma({parameter}) = {sigma:.6g}')
    print(
        f"  90% sky localization = {result['sky_localization_90_deg2']:.6g} deg^2 "
        f"({result['sky_localization_90_sr']:.6g} sr)"
    )
    master_rows.append({
        'network': network_name, 'network_snr': result['network_snr'],
        'rank': result['rank'],
        'sky_localization_90_sr': result['sky_localization_90_sr'],
        'sky_localization_90_deg2': result['sky_localization_90_deg2'],
        **{f'sigma_{parameter}': sigma for parameter, sigma in zip(FISHER_PARAMETERS, result['uncertainties'])},
    })

master_summary_path = DATADIR / 'gw150914_amigo_network_fisher_summary.csv'
columns = list(master_rows[0])
with master_summary_path.open('w') as handle:
    handle.write(','.join(columns) + '\n')
    for row in master_rows:
        handle.write(','.join(str(row[column]) for column in columns) + '\n')
print(f'\nSaved master summary to {master_summary_path}')

AMIGO ($L=10^7$ m; LISA response): SNR=36.285


AMIGO-5 ($L=5\times10^7$ m; LISA response): SNR=52.401


LIGO Hanford O5: SNR=186.851


LIGO Livingston O5: SNR=218.006


Virgo O5: SNR=115.654


...

CE-LHO: SNR=4187.160


CE-LLO: SNR=4885.699


LISA: SNR=5.592


Taiji (LISA response): SNR=13.217


TianQin (LISA response): SNR=9.371

AMIGO: SNR=36.285, rank=11/11
  AMIGO ($L=10^7$ m; LISA response): SNR=36.285
  sigma(chirp_mass) = 2.20125e-05
  sigma(mass_ratio) = 0.0352044
  sigma(luminosity_distance) = 37.8456
  sigma(theta_jn) = 0.254209
  sigma(ra) = 0.00757764
  sigma(dec) = 0.00515377
  sigma(psi) = 0.273715
  sigma(geocent_time) = 0.0374143
  sigma(phase) = 2.76091
  sigma(a_1) = 1.82156
  sigma(a_2) = 2.36523
  90% sky localization = 1.0568 deg^2 (0.000321919 sr)

AMIGO-5: SNR=52.401, rank=10/11
  AMIGO-5 ($L=5\times10^7$ m; LISA response): SNR=52.401
  sigma(chirp_mass) = 5.17136e-06
  sigma(mass_ratio) = 0.00730712
  sigma(luminosity_distance) = 16.9374
  sigma(theta_jn) = 0.106036
  sigma(ra) = 0.00460486
  sigma(dec) = 0.00498065
  sigma(psi) = 0.151173
  sigma(geocent_time) = 0.028074
  sigma(phase) = 0.474323
  sigma(a_1) = 0.000105667
  sigma(a_2) = 0.000137882
  90% sky localization = 1.01462 deg^2 (0.00030907 sr)

AMIGO + LVK: SNR=315.882, rank=11/11
  AMIGO ($L

## Notes

- These are local Gaussian Fisher forecasts using `IMRPhenomXPHM`.
- AMIGO and AMIGO-5 use the GWFish LISA solar-orbit response as a response approximation; the repository AMIGO strain PSD and AMIGO frequency band replace the LISA PSD/grid.
- Taiji and TianQin also reuse the LISA response, matching the approximation used in the companion notebook.
- LVK uses GWFish's built-in LHO, LLO, Virgo, and KAGRA configurations. The two CE detectors use the CE2 PSD at LHO and LLO sites.
- No corner plots are generated; all requested numerical Fisher, covariance, uncertainty, rank, SNR, and 90% sky-localization results are saved under `data/gw150914_gwfish_fisher_amigo/`.